In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import time

from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

import lightgbm as lgb
from lightgbm import LGBMClassifier

pd.set_option('display.max_columns', 200)

mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("loan-default")

print(f"LightGBM version: {lgb.__version__}")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

In [ ]:
df = pd.read_parquet('../data/loans_clean.parquet')

# EDA cleaning
df['dti'] = df['dti'].replace([-1, 999], np.nan).clip(upper=100)
df['credit_history_months'] = df['credit_history_months'].replace(999, np.nan)
df['annual_inc'] = df['annual_inc'].clip(upper=1_000_000)
df['revol_util'] = df['revol_util'].clip(upper=100)

# Temporal split
df_train = df[df['issue_year'].isin([2014, 2015])].copy()
df_val   = df[df['issue_year'] == 2016].copy()
df_test  = df[df['issue_year'] == 2017].copy()

y_train = df_train['default'].values
y_val   = df_val['default'].values

print(f"Train: {len(df_train):,}  |  Val: {len(df_val):,}  |  Test: {len(df_test):,}")

GAIN_RATE = 0.10
LOSS_RATE = 0.50

def calculate_profit(df_subset, approval_mask):
    approved = df_subset[approval_mask]
    if len(approved) == 0:
        return 0
    profit = np.where(
        approved['default'] == 0,
         GAIN_RATE * approved['loan_amnt'],
        -LOSS_RATE * approved['loan_amnt']
    )
    return profit.sum()

In [ ]:
def prep_features(df):
    df = df.copy()
    redundant = ['fico_range_high', 'funded_amnt', 'funded_amnt_inv',
                 'num_sats', 'installment', 'num_rev_tl_bal_gt_0']
    joint_cols = [c for c in df.columns if c.startswith('sec_app_') or c.endswith('_joint')]
    high_cardinality = ['zip_code', 'sub_grade']
    split_cols = ['issue_year']
    cols_to_drop = redundant + joint_cols + high_cardinality + split_cols
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    
    emp_map = {f'{i} year{"s" if i != 1 else ""}': i for i in range(1, 10)}
    emp_map['< 1 year'] = 0
    emp_map['10+ years'] = 10
    df['emp_length'] = df['emp_length'].map(emp_map)
    return df


def build_preprocessor(numeric_cols, categorical_cols):
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    return ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_cols),
            ('cat', categorical_transformer, categorical_cols)
        ]
    )


df_train_fe = prep_features(df_train)
df_val_fe   = prep_features(df_val)

X_train = df_train_fe.drop(columns=['default'])
X_val   = df_val_fe.drop(columns=['default'])

X_train['term'] = X_train['term'].str.extract(r'(\d+)').astype(int)
X_val['term']   = X_val['term'].str.extract(r'(\d+)').astype(int)

categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

print(f"Train shape: {X_train.shape}")
print(f"Categorical: {len(categorical_cols)} cols, Numeric: {len(numeric_cols)} cols")

In [ ]:
# Fit preprocessor on train, transform both
preprocessor_lgb = build_preprocessor(numeric_cols, categorical_cols)
preprocessor_lgb.fit(X_train)

X_train_proc = preprocessor_lgb.transform(X_train)
X_val_proc   = preprocessor_lgb.transform(X_val)

print(f"After preprocessing: train {X_train_proc.shape}, val {X_val_proc.shape}")

lgbm = LGBMClassifier(
    n_estimators=1000,          # max; early stopping will cut it short
    learning_rate=0.05,         # smaller = more trees needed but better generalization
    num_leaves=31,              # default; controls complexity
    min_child_samples=20,       # regularization (min samples per leaf)
    random_state=42,
    n_jobs=-1,
    verbose=-1                  # suppress LGBM's chatty output
)

with mlflow.start_run(run_name="lgbm_baseline"):
    mlflow.log_param("model_type", "LGBMClassifier")
    mlflow.log_param("n_estimators_max", 1000)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("num_leaves", 31)
    mlflow.log_param("min_child_samples", 20)
    mlflow.log_param("early_stopping_rounds", 50)
    mlflow.log_param("class_weight", "None")
    mlflow.log_param("n_features_input", X_train_proc.shape[1])
    
    print("Training LightGBM with early stopping...")
    start = time.time()
    lgbm.fit(
        X_train_proc, y_train,
        eval_set=[(X_val_proc, y_val)],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    train_time = time.time() - start
    
    # Capture how many iterations actually ran
    actual_iters = lgbm.best_iteration_ if hasattr(lgbm, 'best_iteration_') else 1000
    mlflow.log_metric("actual_iterations", actual_iters)
    
    val_probs = lgbm.predict_proba(X_val_proc)[:, 1]
    
    auc = roc_auc_score(y_val, val_probs)
    pr_auc = average_precision_score(y_val, val_probs)
    brier = brier_score_loss(y_val, val_probs)
    
    thresholds = np.linspace(0.05, 0.55, 100)
    profits = [calculate_profit(df_val, val_probs < t) for t in thresholds]
    best_idx = int(np.argmax(profits))
    best_threshold = float(thresholds[best_idx])
    best_profit = float(profits[best_idx])
    approval_rate = float((val_probs < best_threshold).mean())
    
    mlflow.log_param("best_threshold", best_threshold)
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("brier", brier)
    mlflow.log_metric("profit_at_threshold", best_profit)
    mlflow.log_metric("approval_rate", approval_rate)
    mlflow.log_metric("train_time_seconds", train_time)
    
    mlflow.sklearn.log_model(lgbm, "model")
    
    print(f"\n=== LIGHTGBM (baseline) ===")
    print(f"Train time:     {train_time:.1f}s")
    print(f"Iterations:     {actual_iters} (max 1000)")
    print(f"AUC:            {auc:.4f}")
    print(f"PR-AUC:         {pr_auc:.4f}")
    print(f"Brier:          {brier:.4f}")
    print(f"Best threshold: {best_threshold:.4f}")
    print(f"Approval rate:  {approval_rate:.2%}")
    print(f"Profit:         ${best_profit:,.0f}")
    
    print(f"\n--- Full scoreboard ---")
    print(f"RF baseline:    AUC 0.7158, Brier 0.1910, Profit $60.4M")
    print(f"RF no_balance:  AUC 0.7157, Brier 0.1609, Profit $60.5M")
    print(f"LightGBM:       AUC {auc:.4f}, Brier {brier:.4f}, Profit ${best_profit/1e6:.1f}M")

In [ ]:
import optuna
from optuna.samplers import TPESampler

# Suppress optuna's per-trial logging noise (we have the progress bar)
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators': 1000,  # max; early stopping decides actual
        'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves':         trial.suggest_int('num_leaves', 15, 127),
        'min_child_samples':  trial.suggest_int('min_child_samples', 5, 100),
        'feature_fraction':   trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction':   trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq':       trial.suggest_int('bagging_freq', 0, 7),
        'lambda_l1':          trial.suggest_float('lambda_l1', 1e-3, 10, log=True),
        'lambda_l2':          trial.suggest_float('lambda_l2', 1e-3, 10, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    
    model = LGBMClassifier(**params)
    model.fit(
        X_train_proc, y_train,
        eval_set=[(X_val_proc, y_val)],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    val_probs = model.predict_proba(X_val_proc)[:, 1]
    return roc_auc_score(y_val, val_probs)


print("Starting Optuna search (50 trials)...")

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler)

start = time.time()
study.optimize(objective, n_trials=50, show_progress_bar=True)
search_time = time.time() - start

print(f"\n=== OPTUNA SEARCH COMPLETE ===")
print(f"Total time: {search_time/60:.1f} min")
print(f"Trials completed: {len(study.trials)}")
print(f"Best AUC found: {study.best_value:.4f}")
print(f"Improvement over baseline (0.7250): {study.best_value - 0.7250:+.4f}")

print(f"\nBest hyperparameters:")
for k, v in study.best_params.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

In [ ]:
best_params = study.best_params.copy()
best_params['n_estimators'] = 1000
best_params['random_state'] = 42
best_params['n_jobs'] = -1
best_params['verbose'] = -1

lgbm_tuned = LGBMClassifier(**best_params)

with mlflow.start_run(run_name="lgbm_tuned"):
    for k, v in best_params.items():
        mlflow.log_param(k, v)
    mlflow.log_param("model_type", "LGBMClassifier")
    mlflow.log_param("optuna_n_trials", 50)
    mlflow.log_param("early_stopping_rounds", 50)
    
    print("Training final LightGBM with tuned hyperparameters...")
    start = time.time()
    lgbm_tuned.fit(
        X_train_proc, y_train,
        eval_set=[(X_val_proc, y_val)],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    train_time = time.time() - start
    
    actual_iters = lgbm_tuned.best_iteration_ if hasattr(lgbm_tuned, 'best_iteration_') else 1000
    mlflow.log_metric("actual_iterations", actual_iters)
    
    val_probs = lgbm_tuned.predict_proba(X_val_proc)[:, 1]
    
    auc = roc_auc_score(y_val, val_probs)
    pr_auc = average_precision_score(y_val, val_probs)
    brier = brier_score_loss(y_val, val_probs)
    
    thresholds = np.linspace(0.05, 0.55, 100)
    profits = [calculate_profit(df_val, val_probs < t) for t in thresholds]
    best_idx = int(np.argmax(profits))
    best_threshold = float(thresholds[best_idx])
    best_profit = float(profits[best_idx])
    approval_rate = float((val_probs < best_threshold).mean())
    
    mlflow.log_param("best_threshold", best_threshold)
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("brier", brier)
    mlflow.log_metric("profit_at_threshold", best_profit)
    mlflow.log_metric("approval_rate", approval_rate)
    mlflow.log_metric("train_time_seconds", train_time)
    
    mlflow.sklearn.log_model(lgbm_tuned, "model")
    
    print(f"\n=== LIGHTGBM TUNED ===")
    print(f"Train time:     {train_time:.1f}s")
    print(f"Iterations:     {actual_iters}")
    print(f"AUC:            {auc:.4f}")
    print(f"PR-AUC:         {pr_auc:.4f}")
    print(f"Brier:          {brier:.4f}")
    print(f"Best threshold: {best_threshold:.4f}")
    print(f"Approval rate:  {approval_rate:.2%}")
    print(f"Profit:         ${best_profit:,.0f}")
    
    print(f"\n--- Comparison to LightGBM baseline ---")
    print(f"LGBM baseline: AUC 0.7250, Brier 0.1585, Profit $63.7M")
    print(f"LGBM tuned:    AUC {auc:.4f}, Brier {brier:.4f}, Profit ${best_profit/1e6:.1f}M")

In [ ]:
import json
print(json.dumps(study.best_params, indent=2))